# Train YOLOv8n-seg on a synthetic dataset

Before running: on the Mac, zip the dataset (done by the pipeline: `data/yolo/<DATASET>.zip`) and upload it to Google Drive as `MyDrive/sdf/<DATASET>.zip`.
Runtime → Change runtime type → **T4 GPU**. Results (weights, plots, MLflow logs) are saved to `MyDrive/sdf/`, so they survive a disconnect.
Keep the laptop lid open and plugged in while training (a closed lid stopped the first run at epoch 28).

Steps: 1) settings + Drive, 2) install, 3) unpack data, 4) 1-epoch smoke test, 5) full training. If the session drops: run 1-3, then the resume cell.

In [ ]:
# 1) Settings + Google Drive
DATASET = 'synth_v2'          # which zip in MyDrive/sdf/ to train on (synth_v1, synth_v2, ...)
RUN = f'{DATASET}_baseline'   # run name -> MyDrive/sdf/runs/<RUN>/
from google.colab import drive
drive.mount('/content/drive')
SDF = '/content/drive/MyDrive/sdf'

In [ ]:
# 2) Install: ultralytics pinned (same version as on the Mac); mlflow optional.
#    mlflow==2.17.2 had no wheel for Colab's newer Python and failed to build (2026-10-17).
!pip install -q ultralytics==8.3.40
!pip install -q mlflow || echo "mlflow could not be installed, continuing without it"
import os
from ultralytics import YOLO, settings
try:
    import mlflow  # experiment tracking (optional)
    os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'  # newer mlflow refuses the folder backend otherwise
    os.environ['MLFLOW_TRACKING_URI'] = f'file:{SDF}/mlruns'
    os.environ['MLFLOW_EXPERIMENT_NAME'] = 'sim2real_scratch'
    settings.update({'mlflow': True})
    print('mlflow on')
except ImportError:
    settings.update({'mlflow': False})
    print('mlflow off (training results are still saved as results.csv in Drive)')

In [ ]:
# 3) Unpack the dataset to the fast local disk (reading from Drive during training is slow)
!mkdir -p /content/yolo && unzip -qo {SDF}/{DATASET}.zip -d /content/yolo
!ls /content/yolo/{DATASET}/images/train | wc -l   # expect 4500
!ls /content/yolo/{DATASET}/images/val | wc -l     # expect 500
open('/content/data.yaml', 'w').write(
    f'path: /content/yolo/{DATASET}\ntrain: images/train\nval: images/val\nnames:\n  0: scratch\n')

In [ ]:
# 4) Smoke test: 1 epoch on 5% of the data. Only checks that everything runs.
YOLO('yolov8n-seg.pt').train(data='/content/data.yaml', epochs=1, imgsz=512,
                             fraction=0.05, project=f'{SDF}/runs', name='smoke', exist_ok=True)

In [ ]:
# 5) Full training (same settings for every dataset, so results are comparable). ~1 h on a T4.
#    patience: stop early if val metrics do not improve for 15 epochs.
model = YOLO('yolov8n-seg.pt')  # COCO-pretrained (transfer learning)
model.train(data='/content/data.yaml', epochs=60, imgsz=512, batch=32, seed=0,
            patience=15, project=f'{SDF}/runs', name=RUN, exist_ok=True)
# best weights: MyDrive/sdf/runs/<RUN>/weights/best.pt

In [ ]:
# 6) Only if the session dropped during step 5: continue from the last saved epoch
model = YOLO(f'{SDF}/runs/{RUN}/weights/last.pt')
model.train(resume=True)